Use Case: AI-Powered CV/Resume Review ATS Agent

Problem Statement

Recruiters spend 6-8 seconds scanning each resume. An ATS (Applicant Tracking System) agent can:

Parse and extract structured info from resumes
Score candidates against job descriptions
Check for keyword matches and gaps
Provide actionable feedback to candidates
Rank multiple candidates for a role
Architecture

Resume + Job Description
        ↓
   [Parser Agent] → Extracts structured data
        ↓
   [Keyword Matcher] → Checks JD keyword coverage
        ↓
   [Scorer Agent] → Scores fit (0-100)
        ↓
   [Feedback Agent] → Generates improvement suggestions
        ↓
   Final ATS Report
Built with: LangChain + LangGraph + Anthropic Claude

In [1]:
!pip install -q langchain langchain-anthropic langgraph pydantic python-dotenv

In [2]:
import json
from dotenv import load_dotenv

load_dotenv()  # Loads ANTHROPIC_API_KEY from .env

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import TypedDict, Optional

llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

In [3]:
SAMPLE_RESUME = """
PRIYA SHARMA
Email: priya.sharma@email.com | Phone: +91-9876543210
LinkedIn: linkedin.com/in/priyasharma | Location: Bangalore, India

SUMMARY
Passionate software engineer with 4 years of experience in building scalable web applications 
and microservices. Strong background in Python, cloud technologies, and agile development.

EXPERIENCE

Software Engineer II | Infosys | Jan 2022 - Present
- Developed RESTful APIs using Python Flask and FastAPI serving 50K+ daily requests
- Implemented CI/CD pipelines using Jenkins and GitHub Actions
- Migrated monolithic application to microservices architecture on AWS
- Reduced API response time by 40% through Redis caching and query optimization
- Collaborated with cross-functional teams in Agile/Scrum environment

Software Engineer | TCS | Jul 2020 - Dec 2021
- Built full-stack web applications using React.js and Node.js
- Designed and maintained PostgreSQL and MongoDB databases
- Wrote unit and integration tests achieving 85% code coverage
- Participated in code reviews and mentored 2 junior developers

EDUCATION
B.Tech in Computer Science | VIT University | 2016-2020 | CGPA: 8.5/10

SKILLS
Languages: Python, JavaScript, TypeScript, SQL
Frameworks: Flask, FastAPI, React.js, Node.js, Django
Cloud: AWS (EC2, S3, Lambda, RDS), Docker, Kubernetes
Databases: PostgreSQL, MongoDB, Redis
Tools: Git, Jenkins, GitHub Actions, JIRA, Confluence
Concepts: Microservices, REST APIs, CI/CD, Agile/Scrum, TDD

CERTIFICATIONS
- AWS Certified Solutions Architect - Associate (2023)
- Python Professional Certificate - Coursera (2022)
"""

SAMPLE_JD = """
JOB TITLE: Senior Backend Engineer

COMPANY: FinTech Startup (Series B)

LOCATION: Bangalore, India (Hybrid)

ABOUT THE ROLE:
We are looking for a Senior Backend Engineer to lead the development of our core payment 
processing platform. You'll work with a small, high-impact team building systems that handle 
millions of transactions daily.

REQUIREMENTS:
Must Have:
- 5+ years of experience in backend development
- Strong proficiency in Python (Django or FastAPI preferred)
- Experience with cloud platforms (AWS or GCP)
- Solid understanding of microservices architecture
- Experience with SQL and NoSQL databases
- Knowledge of message queues (Kafka, RabbitMQ, or SQS)
- Experience with containerization (Docker, Kubernetes)
- Strong understanding of API design and RESTful services

Nice to Have:
- Experience in FinTech or payment systems
- Knowledge of event-driven architecture
- Experience with Terraform or Infrastructure as Code
- Familiarity with GraphQL
- Experience leading or mentoring engineers
- Performance optimization experience

SKILLS:
Python, Django, FastAPI, AWS/GCP, PostgreSQL, MongoDB, Redis, Kafka, Docker, 
Kubernetes, Terraform, CI/CD, Microservices, System Design
"""

print(f"Resume length: {len(SAMPLE_RESUME)} chars")
print(f"Job Description length: {len(SAMPLE_JD)} chars")

Resume length: 1579 chars
Job Description length: 1197 chars


In [4]:
class ParsedResume(BaseModel):
    name: str = Field(description="Candidate's full name")
    email: str = Field(description="Email address")
    phone: str = Field(description="Phone number")
    location: str = Field(description="Location/city")
    total_experience_years: float = Field(description="Total years of professional experience")
    current_role: str = Field(description="Current or most recent job title")
    current_company: str = Field(description="Current or most recent company")
    skills: list[str] = Field(description="List of all technical skills mentioned")
    education: str = Field(description="Highest education qualification")
    certifications: list[str] = Field(description="List of certifications")
    key_achievements: list[str] = Field(description="Top 3-5 notable achievements")

class KeywordAnalysis(BaseModel):
    matched_keywords: list[str] = Field(description="JD keywords found in the resume")
    missing_keywords: list[str] = Field(description="JD keywords NOT found in the resume")
    match_percentage: float = Field(description="Percentage of JD keywords matched")
    critical_gaps: list[str] = Field(description="Must-have requirements that are missing")

class ATSScore(BaseModel):
    overall_score: int = Field(description="Overall ATS score out of 100")
    experience_score: int = Field(description="Experience match score out of 25")
    skills_score: int = Field(description="Skills match score out of 25")
    education_score: int = Field(description="Education relevance score out of 15")
    keyword_score: int = Field(description="Keyword optimization score out of 20")
    formatting_score: int = Field(description="Resume formatting/structure score out of 15")
    reasoning: str = Field(description="Brief explanation of the scoring")

print("Pydantic models defined: ParsedResume, KeywordAnalysis, ATSScore")
     

Pydantic models defined: ParsedResume, KeywordAnalysis, ATSScore


In [5]:
class ATSState(TypedDict):
    resume: str
    job_description: str
    parsed_resume: Optional[str]
    keyword_analysis: Optional[str]
    ats_score: Optional[str]
    feedback: Optional[str]
    final_report: Optional[str]

# Node 1: Parse Resume
def parse_resume(state: ATSState) -> dict:
    parser = JsonOutputParser(pydantic_object=ParsedResume)
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert resume parser. Extract structured information from the resume.
{format_instructions}"""),
        ("human", "Parse this resume:\n\n{resume}")
    ])
    chain = prompt | llm | parser
    result = chain.invoke({
        "resume": state["resume"],
        "format_instructions": parser.get_format_instructions()
    })
    print("✅ Resume parsed successfully")
    return {"parsed_resume": json.dumps(result, indent=2)}

# Node 2: Keyword Analysis
def analyze_keywords(state: ATSState) -> dict:
    parser = JsonOutputParser(pydantic_object=KeywordAnalysis)
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an ATS keyword analyzer. Compare the resume against the job description.
Identify which required keywords/skills are present and which are missing.
{format_instructions}"""),
        ("human", """Job Description:
{job_description}

Resume:
{resume}

Parsed Resume Data:
{parsed_resume}""")
    ])
    chain = prompt | llm | parser
    result = chain.invoke({
        "job_description": state["job_description"],
        "resume": state["resume"],
        "parsed_resume": state["parsed_resume"],
        "format_instructions": parser.get_format_instructions()
    })
    print(f"✅ Keyword analysis complete — {result['match_percentage']}% match")
    return {"keyword_analysis": json.dumps(result, indent=2)}

# Node 3: ATS Scoring
def score_resume(state: ATSState) -> dict:
    parser = JsonOutputParser(pydantic_object=ATSScore)
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an ATS scoring engine. Score the resume against the job description.
Be fair but realistic. Consider experience gaps, skill mismatches, and keyword coverage.
{format_instructions}"""),
        ("human", """Job Description:
{job_description}

Parsed Resume:
{parsed_resume}

Keyword Analysis:
{keyword_analysis}""")
    ])
    chain = prompt | llm | parser
    result = chain.invoke({
        "job_description": state["job_description"],
        "parsed_resume": state["parsed_resume"],
        "keyword_analysis": state["keyword_analysis"],
        "format_instructions": parser.get_format_instructions()
    })
    print(f"✅ ATS Score: {result['overall_score']}/100")
    return {"ats_score": json.dumps(result, indent=2)}

# Node 4: Generate Feedback
def generate_feedback(state: ATSState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a career coach and resume expert. Based on the ATS analysis, provide 
actionable feedback to help the candidate improve their resume for this specific role.

Structure your feedback as:
1. **Overall Assessment** (2-3 sentences)
2. **Strengths** (3-5 bullet points)
3. **Areas for Improvement** (3-5 bullet points with specific suggestions)
4. **Missing Keywords to Add** (list specific terms to include)
5. **Recommended Resume Changes** (concrete action items)"""),
        ("human", """Job Description:
{job_description}

Parsed Resume:
{parsed_resume}

Keyword Analysis:
{keyword_analysis}

ATS Score:
{ats_score}""")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({
        "job_description": state["job_description"],
        "parsed_resume": state["parsed_resume"],
        "keyword_analysis": state["keyword_analysis"],
        "ats_score": state["ats_score"]
    })
    print("✅ Feedback generated")
    return {"feedback": result}

# Node 5: Compile Final Report
def compile_report(state: ATSState) -> dict:
    parsed = json.loads(state["parsed_resume"])
    keywords = json.loads(state["keyword_analysis"])
    score = json.loads(state["ats_score"])
    
    report = f"""
{'='*60}
           ATS ANALYSIS REPORT
{'='*60}

📋 CANDIDATE: {parsed['name']}
📧 Email: {parsed['email']}
📍 Location: {parsed['location']}
💼 Current: {parsed['current_role']} at {parsed['current_company']}
📅 Experience: {parsed['total_experience_years']} years

{'─'*60}
📊 ATS SCORE: {score['overall_score']}/100
{'─'*60}
  Experience Match:    {score['experience_score']}/25
  Skills Match:        {score['skills_score']}/25
  Education:           {score['education_score']}/15
  Keyword Coverage:    {score['keyword_score']}/20
  Formatting:          {score['formatting_score']}/15

💡 Scoring Reasoning: {score['reasoning']}

{'─'*60}
🔑 KEYWORD ANALYSIS
{'─'*60}
  Match Rate: {keywords['match_percentage']}%
  
  ✅ Matched: {', '.join(keywords['matched_keywords'])}
  
  ❌ Missing: {', '.join(keywords['missing_keywords'])}
  
  🚨 Critical Gaps: {', '.join(keywords['critical_gaps'])}

{'─'*60}
📝 DETAILED FEEDBACK
{'─'*60}
{state['feedback']}

{'='*60}
"""
    print("✅ Final report compiled")
    return {"final_report": report}

print("All agent nodes defined.")

All agent nodes defined.


In [6]:
# Build the ATS Agent graph
graph = StateGraph(ATSState)

# Add nodes = Agent
graph.add_node("parse_resume", parse_resume)
graph.add_node("analyze_keywords", analyze_keywords)
graph.add_node("score_resume", score_resume)
graph.add_node("generate_feedback", generate_feedback)
graph.add_node("compile_report", compile_report)

# Define the flow
graph.add_edge(START, "parse_resume")
graph.add_edge("parse_resume", "analyze_keywords")
graph.add_edge("analyze_keywords", "score_resume")
graph.add_edge("score_resume", "generate_feedback")
graph.add_edge("generate_feedback", "compile_report")
graph.add_edge("compile_report", END)

# Compile
ats_agent = graph.compile()

print("ATS Agent graph compiled!")
print("Flow: Parse → Keywords → Score → Feedback → Report")

ATS Agent graph compiled!
Flow: Parse → Keywords → Score → Feedback → Report


In [7]:
# Run the full ATS pipeline
result = ats_agent.invoke({
    "resume": SAMPLE_RESUME,
    "job_description": SAMPLE_JD,
})

print(result["final_report"])

✅ Resume parsed successfully
✅ Keyword analysis complete — 48.0% match
✅ ATS Score: 62/100
✅ Feedback generated
✅ Final report compiled

           ATS ANALYSIS REPORT

📋 CANDIDATE: Priya Sharma
📧 Email: priya.sharma@email.com
📍 Location: Bangalore, India
💼 Current: Software Engineer II at Infosys
📅 Experience: 4 years

────────────────────────────────────────────────────────────
📊 ATS SCORE: 62/100
────────────────────────────────────────────────────────────
  Experience Match:    15/25
  Skills Match:        18/25
  Education:           12/15
  Keyword Coverage:    10/20
  Formatting:          7/15

💡 Scoring Reasoning: Priya has 4 years of experience, which is less than the required 5+ years, impacting the experience score. She possesses many of the required skills, like Python, FastAPI, AWS, microservices, and CI/CD, but lacks experience with message queues and has not mentioned working with Django or leading teams, affecting the skills and keyword scores. Her education is relevant